In [1]:

import json
import random
import string
from pathlib import Path
from typing import List, Tuple, Dict

from faker import Faker
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import inch
from reportlab.lib.utils import simpleSplit
from reportlab.pdfbase.pdfmetrics import stringWidth
import fitz  # PyMuPDF

In [2]:

fake = Faker("es_MX")
random.seed(4321)

A4_W, A4_H = A4
FONT = "Helvetica"
FS = 8.6        # un poco más chico para dar aire
FS_B = 10
LEFT = 0.65*inch
RIGHT = A4_W - 0.65*inch
HEADER_H = 0.36*inch
TOP = A4_H - 0.75*inch
ROW_MIN = 0.35*inch
CELL_PAD_X = 4          # padding horizontal (pt)
CELL_PAD_RIGHT = 6      # padding derecha (pt)
MAX_CONC_LINES = 4

MESES = ["ENE","FEB","MAR","ABR","MAY","JUN","JUL","AGO","SEP","OCT","NOV","DIC"]

def fecha_mx():
    d = random.randint(1,28); m = random.randint(1,12)
    return f"{d:02d} {MESES[m-1]}"

def rd(n): return "".join(random.choices("0123456789", k=n))
def ra(n): return "".join(random.choices(string.ascii_uppercase+"0123456789", k=n))
def peso(x): return f"${x:,.2f}"

def concepto_largo():
    base = random.choice([
        "GRUPO GASOLINERO TH","SERVICIO CHRISPTO OP AV",
        "TRANSF INTERBANCARIA SPEI","ABC-MART OSAKA JP",
        "HOTEL WORLDWIDE MEXICO S DE","UBER TRIP",
        "AMAZON MKTPLACE","RETIRO CAJERO AUTOMATICO RED",
        "NOMINA " + fake.company().upper()
    ])
    extra=[]
    if random.random()<.70: extra.append(f"REF {rd(10)} AUT {ra(6)}")
    if random.random()<.45: extra.append(f"MONTO ORIGEN {random.randint(1,199)}.{random.randint(0,99):02d} USD")
    if random.random()<.55: extra.append(f"CUENTA {rd(11)}")
    return base + "  " + "  ".join(extra)

def origen_ref():
    s = "0"*12 + rd(6)
    if random.random()<.4: s += " / " + rd(4)
    return s

# ---------------- utilidades de dibujo seguras ----------------
def ellipsize(cadena: str, max_width_pt: float, font=FONT, size=FS) -> str:
    """Corta con '…' si no cabe en max_width_pt."""
    if stringWidth(cadena, font, size) <= max_width_pt:
        return cadena
    dots_w = stringWidth("…", font, size)
    out = []
    for ch in cadena:
        if stringWidth("".join(out)+ch, font, size) + dots_w > max_width_pt:
            break
        out.append(ch)
    return "".join(out) + "…"

def draw_text_in_cell(c, x_left, x_right, y_baseline, text, align="left", font=FONT, size=FS):
    """Respeta límites de celda con padding + ellipsis."""
    xL = x_left + CELL_PAD_X
    xR = x_right - CELL_PAD_RIGHT
    max_w = max(0, xR - xL)
    txt = ellipsize(text, max_w, font, size)
    c.setFont(font, size)
    if align == "left":
        c.drawString(xL, y_baseline, txt)
    else:  # right
        w = stringWidth(txt, font, size)
        c.drawString(xR - w, y_baseline, txt)

def draw_multiline_in_cell(c, x_left, x_right, y_baseline, text, max_lines=3, font=FONT, size=FS, line_h=None):
    xL = x_left + CELL_PAD_X
    xR = x_right - CELL_PAD_RIGHT
    max_w = max(0, xR - xL)
    if line_h is None:
        line_h = size * 1.28
    lines = simpleSplit(text, font, size, max_w)[:max_lines]
    y = y_baseline
    c.setFont(font, size)
    for ln in lines:
        c.drawString(xL, y, ln)
        y -= line_h
    used_h = max(ROW_MIN, (len(lines) * line_h + 6))
    return used_h, lines

def _draw_right(c, x_right, y, text):
    draw_text_in_cell(c, x_right-120, x_right, y, text, align="right")

# ---------------- layout columnas (con normalización de ancho) -------------
def compute_columns(desired_in_inches):
    """Escala los anchos 'deseados' para que quepan entre LEFT y RIGHT."""
    avail_in = (RIGHT - LEFT) / inch
    total_desired = sum(desired_in_inches)
    scale = min(1.0, avail_in / total_desired)
    widths_in = [w*scale for w in desired_in_inches]
    xs = [LEFT]
    for w in widths_in[:-1]:
        xs.append(xs[-1] + w*inch)
    rights = [x + w*inch for x, w in zip(xs, widths_in)]
    return xs, rights, widths_in

# Config deseada (se escala automáticamente si no cabe):
DESIRED_COL_W = (0.90, 3.30, 0.90, 0.75, 0.75, 0.95)  # Fecha, Concepto, Origen, Depósito, Retiro, Saldo
COL_X, COL_R, COL_W_IN = compute_columns(DESIRED_COL_W)
FECHA, CONC, ORI, DEP, RET, SAL = range(6)

# ---------------- encabezado / marco ----------------
def draw_top_and_header(c, hide_prob=0.22):
    # Banda
    c.setLineWidth(1.0); c.setStrokeGray(0); c.setFillGray(0.92)
    c.rect(LEFT, TOP-HEADER_H, RIGHT-LEFT, HEADER_H, stroke=1, fill=1)
    c.setFillGray(0); c.setFont("Helvetica-Bold", 11)
    c.drawString(LEFT+8, TOP-HEADER_H+8, "Detalle de tus movimientos")
    # Header
    y = TOP - HEADER_H - 0.08*inch
    c.setLineWidth(0.8); c.line(LEFT, y, RIGHT, y)
    c.setFont("Helvetica-Bold", FS_B)
    labels = ["Fecha","Concepto","Origen / Referencia","Depósito","Retiro","Saldo"]
    for i, lab in enumerate(labels):
        if random.random() > hide_prob:
            c.drawString(COL_X[i] + CELL_PAD_X, y - 0.22*inch, lab)
    c.line(LEFT, y - 0.30*inch, RIGHT, y - 0.30*inch)
    return y - 0.46*inch  # baseline inicial

# ---------------- GT (sin ORIGEN) ----------------
def build_gt(movs: List[Dict]) -> Dict:
    def norm(s): return 0.0 if not s else float(s.replace("$","").replace(",",""))
    dep = sum(norm(m.get("deposito","")) for m in movs)
    ret = sum(norm(m.get("retiro","")) for m in movs)
    saldo_final = movs[-1]["saldo"] if movs else ""
    return {
        "tabla":"movimientos_scotia_like",
        "movimientos":[{k:m[k] for k in["fecha","concepto","deposito","retiro","saldo"]} for m in movs],
        "totales":{"depositos":peso(dep),"retiros":peso(ret),"saldo_final":saldo_final}
    }

# ---------------- generador de página ----------------
def generar_pagina(c: canvas.Canvas, filas_max=16) -> List[Dict]:
    baseline = draw_top_and_header(c, hide_prob=0.25)
    movs=[]; saldo = random.uniform(300000, 980000)
    line_h = FS*1.28
    for i in range(filas_max):
        if baseline < 1.1*inch: break
        # sample
        fecha = fecha_mx(); conc = concepto_largo(); ori = origen_ref()
        is_dep = random.random()<0.45
        dep = round(random.uniform(1200, 60000),2) if is_dep else 0.0
        ret = 0.0 if is_dep else round(random.uniform(150, 9000),2)
        saldo = saldo + dep - ret
        # zebra
        if i % 2 == 1:
            c.setFillGray(0.965)
            c.rect(LEFT, baseline + 0.16*inch, RIGHT-LEFT, -ROW_MIN, stroke=0, fill=1)
            c.setFillGray(0)
        # fecha
        draw_text_in_cell(c, COL_X[FECHA], COL_R[FECHA], baseline, fecha, align="left")
        # concepto multilinea (dentro de celda)
        used_h, lines = draw_multiline_in_cell(
            c, COL_X[CONC], COL_R[CONC], baseline, conc,
            max_lines=MAX_CONC_LINES, font=FONT, size=FS, line_h=line_h
        )
        row_h = max(ROW_MIN, used_h)
        # origen (gris + ellipsis dentro de su celda)
        c.setFillGray(0.35)
        draw_text_in_cell(c, COL_X[ORI], COL_R[ORI], baseline, ori, align="left", font=FONT, size=FS-1.2)
        c.setFillGray(0)
        # importes (solo uno por fila) alineados a derecha y contenidos en su celda
        if dep>0: _draw_right(c, COL_R[DEP], baseline, peso(dep))
        if ret>0: _draw_right(c, COL_R[RET], baseline, peso(ret))
        _draw_right(c, COL_R[SAL], baseline, peso(saldo))
        # regla fila
        c.setLineWidth(0.25); c.line(LEFT, baseline-(row_h-0.46*inch), RIGHT, baseline-(row_h-0.46*inch))
        # GT
        movs.append({
            "fecha": fecha,
            "concepto": " ".join(lines),
            "deposito": peso(dep) if dep>0 else "",
            "retiro":  peso(ret) if ret>0 else "",
            "saldo":   peso(saldo),
        })
        baseline -= row_h
    return movs

# ---------------- pipeline de ejemplo ----------------
def pdf_to_pngs(pdf_path: Path, out_dir: Path, prefix: str, dpi=300):
    out_dir.mkdir(parents=True, exist_ok=True)
    doc = fitz.open(str(pdf_path)); mat = fitz.Matrix(dpi/72.0, dpi/72.0)
    outs=[]
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=mat, alpha=False)
        p = out_dir / f"{prefix}_{i:04d}.png"; pix.save(str(p)); outs.append(str(p))
    doc.close(); return outs

def generar_dataset_scotia_sin_desbordes(num_docs=8, out_root="scotia_synth_fix"):
    root = Path(out_root); (root/"pdf").mkdir(parents=True, exist_ok=True); (root/"png").mkdir(parents=True, exist_ok=True)
    pairs=[]
    for i in range(num_docs):
        pdf_path = root/"pdf"/f"scotia_{i:05d}.pdf"
        c = canvas.Canvas(str(pdf_path), pagesize=A4)
        movs = generar_pagina(c, filas_max=16)
        c.showPage(); c.save()
        imgs = pdf_to_pngs(pdf_path, root/"png", pdf_path.stem, dpi=300)
        if imgs:
            pairs.append((imgs[0], build_gt(movs)))
    # JSONL
    random.shuffle(pairs); n = int(len(pairs)*0.9)
    with open(root/"train.jsonl","w",encoding="utf-8") as f:
        for img,gt in pairs[:n]:
            f.write(json.dumps({"image_path":img,"ground_truth":json.dumps({"gt_parse":gt},ensure_ascii=False,separators=(",",":"))},ensure_ascii=False)+"\n")
    with open(root/"validation.jsonl","w",encoding="utf-8") as f:
        for img,gt in pairs[n:]:
            f.write(json.dumps({"image_path":img,"ground_truth":json.dumps({"gt_parse":gt},ensure_ascii=False,separators=(",",":"))},ensure_ascii=False)+"\n")
    print(f"[OK] páginas: {len(pairs)}  train:{n}  val:{len(pairs)-n}  out:{root.resolve()}")

In [4]:
def generar_pagina(c: canvas.Canvas, filas_max=16) -> List[Dict]:
    y_top = draw_top_and_header(c, hide_prob=0.25)  # y_top = parte superior del primer renglón
    movs = []
    saldo = random.uniform(300000, 980000)
    line_h = FS * 1.28

    for i in range(filas_max):
        # si no cabe otra fila, rompe
        if y_top - ROW_MIN < 1.1 * inch:
            break

        # --- datos sintéticos ---
        fecha = fecha_mx()
        conc  = concepto_largo()
        ori   = origen_ref()
        is_dep = random.random() < 0.45
        dep = round(random.uniform(1200, 60000), 2) if is_dep else 0.0
        ret = 0.0 if is_dep else round(random.uniform(150, 9000), 2)
        saldo = saldo + dep - ret

        # alto de fila según líneas de “Concepto”
        max_w = COL_R[CONC] - COL_X[CONC] - 4
        lines = simpleSplit(conc, FONT, FS, max_w)[:MAX_CONC_LINES]
        used_h = max(ROW_MIN, (len(lines) * line_h + 6))
        row_h = used_h  # ¡usa SIEMPRE row_h para todo!

        # ¿cabe completa?
        if y_top - row_h < 1.1 * inch:
            break

        # --- zebra (usa row_h, no ROW_MIN) ---
        if i % 2 == 1:
            c.setFillGray(0.965)
            c.rect(LEFT, y_top, RIGHT - LEFT, -row_h, stroke=0, fill=1)
            c.setFillGray(0)

        # baseline de texto dentro de la fila
        baseline = y_top - 0.20 * inch

        # FECHA
        draw_text_in_cell(c, COL_X[FECHA], COL_R[FECHA], baseline, fecha, align="left")

        # CONCEPTO (multilínea, dentro de celda)
        c.setFont(FONT, FS)
        y_line = baseline
        for ln in lines:
            draw_text_in_cell(c, COL_X[CONC], COL_R[CONC], y_line, ln, align="left")
            y_line -= line_h

        # ORIGEN/REFERENCIA (gris + ellipsis)
        c.setFillGray(0.35)
        draw_text_in_cell(c, COL_X[ORI], COL_R[ORI], baseline, ori, align="left", font=FONT, size=FS - 1.2)
        c.setFillGray(0)

        # IMPORTES (derecha, sin desbordes)
        if dep > 0:
            _draw_right(c, COL_R[DEP], baseline, peso(dep))
        if ret > 0:
            _draw_right(c, COL_R[RET], baseline, peso(ret))
        _draw_right(c, COL_R[SAL], baseline, peso(saldo))

        # regla inferior EXACTAMENTE al final de la fila
        c.setLineWidth(0.25)
        c.line(LEFT, y_top - row_h, RIGHT, y_top - row_h)

        # GT
        movs.append({
            "fecha": fecha,
            "concepto": " ".join(lines),
            "deposito": peso(dep) if dep > 0 else "",
            "retiro":  peso(ret) if ret > 0 else "",
            "saldo":   peso(saldo),
        })

        # avanza cursor al siguiente renglón
        y_top -= row_h

    return movs





In [6]:
generar_dataset_scotia_sin_desbordes(num_docs=5000, out_root="scotia_synth_fix")

[OK] páginas: 5000  train:4500  val:500  out:C:\Users\ovill\OneDrive\Documentos\Donut\Ejercicio Sintetico 3\scotia_synth_fix
